# CPU / Memory Monitoring Agent (MCP + LangGraph)

이 노트북은 **cpu_memory_server.py**에서 제공하는 MCP 도구를 사용해 로컬 시스템의 CPU·메모리 상태를 모니터링하는 LangGraph 에이전트를 구축합니다.

## 학습 목표
- `FastMCP` 기반 커스텀 MCP 서버(cpu_memory_server.py)를 stdio로 연결합니다.
- `MultiServerMCPClient`로 도구를 로드하고 `create_agent`로 ReAct 에이전트를 생성합니다.
- `ToolNode`를 활용한 커스텀 워크플로우로 복합 질의를 처리합니다.

## 제공되는 MCP 도구

| 도구 | 설명 |
|------|------|
| `get_cpu_usage` | 전체 CPU 사용률(%) |
| `get_memory_usage` | RAM 총량 / 사용량 / 가용량 / 사용률 |
| `get_cpu_per_core` | 코어별 CPU 사용률 |
| `get_top_processes` | CPU 사용률 상위 N개 프로세스 |
| `get_system_info` | CPU + RAM + 가동 시간 통합 요약 |

---

## Part 1: 환경 설정

`dotenv`로 API 키를 로드하고, `langchain_teddynote`의 LangSmith 추적을 활성화합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv(override=True)
logging.langsmith("LangGraph-Tutorial")

---

## Part 2: 공통 헬퍼 — Windows stdio 패치 & MCP 클라이언트 설정

Windows + Jupyter 환경에서 `mcp.client.stdio`가 `sys.stderr.fileno()`를 호출해 오류가 발생하는 문제를 패치합니다.  
이후 MCP 클라이언트를 생성하고 도구를 로드하는 헬퍼 함수를 정의합니다.

In [ ]:
import sys
import os
import nest_asyncio
from typing import List, Dict, Any

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient

# Jupyter 환경에서 중첩 비동기 루프를 허용합니다
nest_asyncio.apply()

# Windows + Jupyter: mcp stdio가 Jupyter의 stderr에 fileno()를 호출하면 오류가 발생합니다.
# stderr를 os.devnull로 패치해 이 문제를 우회합니다.
if sys.platform == "win32":
    import mcp.client.stdio as _mcp_stdio

    _devnull_file = open(os.devnull, "w")

    if hasattr(_mcp_stdio.stdio_client, "__wrapped__"):
        _mcp_stdio.stdio_client.__wrapped__.__defaults__ = (_devnull_file,)

    _mcp_stdio._create_platform_compatible_process.__defaults__ = (
        None,
        _devnull_file,
        None,
    )


async def setup_mcp_client(server_configs: dict):
    """MCP 클라이언트를 생성하고 도구 목록을 반환합니다."""
    client = MultiServerMCPClient(server_configs)
    tools = await client.get_tools()
    print(f"[MCP] {len(tools)}개의 도구가 로드되었습니다:")
    for tool in tools:
        print(f"  - {tool.name}")
    return client, tools

---

## Part 3: CPU/Memory MCP 서버 연결

`cpu_memory_server.py`를 **stdio** 방식으로 연결합니다.  
`uv run python cpu_memory_server.py` 명령으로 서버를 서브프로세스로 실행합니다.

In [ ]:
# CPU/Memory MCP 서버 구성 (stdio 전송 방식)
server_configs = {
    "cpu_memory": {
        "command": "uv",
        "args": ["run", "python", "cpu_memory_server.py"],
        "transport": "stdio",
    },
}

client, tools = await setup_mcp_client(server_configs)

---

## Part 4: create_agent로 ReAct 에이전트 생성 및 실행

`create_agent`를 사용해 MCP 도구를 활용하는 ReAct 에이전트를 생성합니다.  
`InMemorySaver`로 대화 상태를 유지하며, `astream_graph`로 응답을 스트리밍합니다.

In [ ]:
from langchain_teddynote.messages import astream_graph, random_uuid
from langchain_core.runnables import RunnableConfig

llm = init_chat_model("claude-sonnet-4-6", temperature=0)

agent = create_agent(
    llm,
    tools,
    checkpointer=InMemorySaver(),
)

### 예제 1 — 전체 CPU 사용률 조회

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

_ = await astream_graph(
    agent,
    inputs={"messages": [("human", "현재 CPU 사용률을 알려주세요.")]},
    config=config,
)

### 예제 2 — 메모리 사용 현황 조회

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

_ = await astream_graph(
    agent,
    inputs={"messages": [("human", "메모리 사용 현황을 알려주세요.")]},
    config=config,
)

### 예제 3 — CPU 상위 5개 프로세스 조회

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

_ = await astream_graph(
    agent,
    inputs={"messages": [("human", "CPU 사용률이 높은 상위 5개 프로세스를 보여주세요.")]},
    config=config,
)

### 예제 4 — 시스템 전체 요약

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

_ = await astream_graph(
    agent,
    inputs={"messages": [("human", "시스템 전체 현황(CPU, 메모리, 가동 시간)을 한눈에 보여주세요.")]},
    config=config,
)

---

## Part 5: ToolNode 기반 커스텀 워크플로우

`StateGraph` + `ToolNode`를 사용해 에이전트-도구 루프를 직접 제어합니다.  
이 방식은 조건부 분기, 병렬 처리 등 복잡한 워크플로우 구현에 적합합니다.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from typing import Annotated, TypedDict


class AgentState(TypedDict):
    """에이전트 상태: 메시지 목록을 누적합니다."""
    messages: Annotated[List[BaseMessage], add_messages]


async def create_cpu_memory_workflow(server_configs: dict):
    """CPU/Memory MCP 도구를 사용하는 ToolNode 기반 워크플로우를 생성합니다."""
    client, tools = await setup_mcp_client(server_configs)

    llm = init_chat_model("claude-sonnet-4-6", temperature=0)
    llm_with_tools = llm.bind_tools(tools)

    workflow = StateGraph(AgentState)

    async def agent_node(state: AgentState):
        """에이전트 노드: LLM을 호출하여 다음 행동을 결정합니다."""
        response = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [response]}

    tool_node = ToolNode(tools)

    workflow.add_node("agent", agent_node)
    workflow.add_node("tools", tool_node)
    workflow.add_edge(START, "agent")
    workflow.add_conditional_edges("agent", tools_condition)
    workflow.add_edge("tools", "agent")

    return workflow.compile(checkpointer=InMemorySaver())

In [ ]:
# ToolNode 워크플로우 생성
mcp_app = await create_cpu_memory_workflow(server_configs)

### 복합 질의 — CPU·메모리 상태 분석 및 이상 프로세스 식별

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

_ = await astream_graph(
    mcp_app,
    inputs={
        "messages": [
            (
                "human",
                "시스템 전체 현황을 확인하고, CPU나 메모리 사용률이 높다면 "
                "원인이 되는 상위 프로세스를 함께 분석해 주세요.",
            )
        ]
    },
    config=config,
)

### 코어별 CPU 사용률 분석

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

_ = await astream_graph(
    mcp_app,
    inputs={
        "messages": [
            (
                "human",
                "각 CPU 코어의 사용률을 확인하고 부하가 고르게 분산되어 있는지 분석해 주세요.",
            )
        ]
    },
    config=config,
)